In [64]:
import logging
import os
import sys
import time

import contextily as ctx
import dask.array as da
import geopandas as gpd
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pygeoutils as pgu
import rasterio
from rasterio import features, windows
from rasterio.transform import from_origin
from scipy.ndimage import distance_transform_edt
from shapely import MultiLineString, line_merge
from shapely.ops import unary_union
from skimage.morphology import medial_axis
from street_reader_funcs import (
    graph_to_geometries,
    skeleton_to_positive_graph_tile,
)
from tqdm import tqdm

In [4]:
# ----------------------------------------------------------------------
# Helper: force plain Python ints (prevents TypeError later)
# ----------------------------------------------------------------------
def _as_int(v):
    return int(float(v))


# ----------------------------------------------------------------------
# Logging configuration (feel free to adjust level)
# ----------------------------------------------------------------------
log = logging.getLogger("centerline")
log.setLevel(logging.INFO)  # DEBUG for more chatter
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(
    logging.Formatter("%(asctime)s %(levelname)-8s %(message)s", "%H:%M:%S")
)
log.addHandler(handler)

log.info("=== START CENTER‑LINE PIPELINE ===")
t_start = time.time()


%matplotlib widget

10:43:28 INFO     === START CENTER‑LINE PIPELINE ===


In [5]:
countries = gpd.read_file("../data/countries.geojson")
country_codes = countries["ISO3166-1-Alpha-2"]

In [6]:
depth = 2
BUFFER_DIST = 100  # meters
tile_size = 16384  # process in 16k×16k pixel tiles

In [ ]:
for code in country_codes:
    if not os.path.exists(
        f"../data/streets/depth_{depth}/streets_{code}_depth{depth}.parquet"
    ):
        log.warning(f"File for country {code} does not exist, skipping.")
        continue
    if os.path.exists(
        f"../data/streets/depth_{depth}/streets_{code}_depth{depth}_centerlines.parquet"
    ):
        log.info(f"Centerlines for country {code} already exist, skipping.")
        continue

    streets = gpd.read_parquet(
        f"../data/streets/depth_{depth}/streets_{code}_depth{depth}.parquet"
    )
    log.info(f"Loaded {len(streets)} street segments for country {code}.")

    streets = streets.set_crs(epsg=4326).to_crs(epsg=3035)

    streets.geometry = streets.geometry.apply(lambda line: line.buffer(BUFFER_DIST))
    merged_buffers = gpd.GeoDataFrame(
        geometry=[unary_union(streets.geometry)], crs=streets.crs
    )

    skel_parts = []
    dist_parts = []
    pixel_size = 10
    global_graph = nx.Graph()

    for idx, feature in merged_buffers.iterrows():
        geom = feature.geometry
        bounds = geom.bounds  # minx, miny, maxx, maxy
        log.debug(f"Feature {idx}: bounds = {bounds}")

        minx, miny, maxx, maxy = bounds
        width = _as_int((maxx - minx) / pixel_size)

        # Create a raster covering the bounding box of the geometry
        height = _as_int((maxy - miny) / pixel_size)
        transform = from_origin(minx, maxy, pixel_size, pixel_size)
        log.info(
            f"Raster grid: {width:,} x {height:,} pixels @ {pixel_size} units/pixel"
        )

        profile = {
            "driver": "GTiff",
            "dtype": "uint8",
            "count": 1,
            "height": height,
            "width": width,
            "crs": merged_buffers.crs,
            "transform": transform,
            "compress": "deflate",
        }

        total_tiles_x = (width + tile_size - 1) // tile_size
        total_tiles_y = (height + tile_size - 1) // tile_size
        total_tiles = total_tiles_x * total_tiles_y
        log.info(
            f"Processing in {total_tiles} tiles of size {tile_size}×{tile_size} pixels"
        )

        with rasterio.MemoryFile() as mem:
            with mem.open(**profile) as dst:
                log.info("Rasterising polygon (tile-wise)…")
                with tqdm(total=total_tiles, desc="Write tiles", unit="tile") as prog:
                    for row_off in range(0, height, tile_size):
                        for col_off in range(0, width, tile_size):
                            win = windows.Window(
                                col_off,  # type: ignore
                                row_off,
                                min(tile_size, width - col_off),
                                min(tile_size, height - row_off),
                            )
                            mask = features.rasterize(
                                [(geom, 1)],
                                out_shape=(int(win.height), int(win.width)),
                                transform=dst.window_transform(win),
                                fill=0,
                                dtype="uint8",
                            )
                            dst.write(mask, 1, window=win)
                            prog.update(1)

            # ------------------------------------------------------------------
            # Lazy skeletonisation (still tile‑wise, never loads full raster)
            # ------------------------------------------------------------------
            log.info("Opening raster for lazy skeletonisation …")
            with mem.open() as src:
                dask_arr = da.from_array(src.read(1), chunks=(tile_size, tile_size))  # type: ignore

                def skel_block(arr):
                    """Thin the binary mask to a 1‑pixel wide skeleton."""
                    return medial_axis(arr.astype(bool), return_distance=False)

                skeleton = da.map_blocks(skel_block, dask_arr, dtype=np.uint8)

            log.info("Writing skeleton raster (tile‑wise)…")
            with rasterio.MemoryFile() as ske_mem:
                with ske_mem.open(**profile) as ske_dst:
                    with tqdm(
                        total=total_tiles, desc="Skeleton tiles", unit="tile"
                    ) as prog:
                        for row_off in range(0, height, tile_size):
                            for col_off in range(0, width, tile_size):
                                win = windows.Window(
                                    col_off,  # type: ignore
                                    row_off,
                                    min(tile_size, width - col_off),
                                    min(tile_size, height - row_off),
                                )
                                block = skeleton[
                                    row_off : row_off + int(win.height),
                                    col_off : col_off + int(win.width),
                                ].compute()

                                dist = distance_transform_edt(block > 0)
                                G_tile = skeleton_to_positive_graph_tile(
                                    block,
                                    dist,
                                    row_off=_as_int(win.row_off),
                                    col_off=_as_int(win.col_off),
                                )

                                global_graph.add_edges_from(G_tile.edges(data=True))
                                global_graph.add_nodes_from(G_tile.nodes(data=True))
                                prog.update(1)

    log.info(
        f"Country {code}: extracted {global_graph.number_of_nodes()} nodes and {global_graph.number_of_edges()} edges."
    )
    gdf_edges = graph_to_geometries(global_graph, transform)
    edges_smoothed = gdf_edges.simplify(tolerance=50, preserve_topology=True)
    gdf_edges_smoothed = gpd.GeoDataFrame(
        gdf_edges.drop(columns="geometry"),
        geometry=edges_smoothed,
        crs=gdf_edges.crs,
    )
    gdf_edges_smoothed.to_parquet(
        f"../data/streets/depth_{depth}/streets_{code}_depth{depth}_centerlines.parquet"
    )

#### Read Centerlines and get Boxes

In [7]:
# get a point every 1000 meters along each linestring
def get_points_along_line(line, distance):
    points = []
    length = line.length
    num_points = int(length // distance)
    for i in range(num_points + 1):
        point = line.interpolate(i * distance)
        points.append(point)
    return points

In [65]:
for i, code in enumerate(country_codes):
    if os.path.exists(
        f"../data/streets/depth_{depth}/streets_{code}_depth{depth}_boxes.parquet"
    ):
        log.info(f"Boxes for country {code} already exist, skipping.")
    if not os.path.exists(
        f"../data/streets/depth_{depth}/streets_{code}_depth{depth}_centerlines.parquet"
    ):
        log.warning(f"File for country {code} does not exist, skipping.")
        continue

    centerlines = gpd.read_parquet(
        f"../data/streets/depth_{depth}/streets_{code}_depth{depth}_centerlines.parquet"
    )
    log.info(f"Loaded {len(centerlines)} center-line segments for country {code}.")

    centerlines_merged = centerlines.geometry.apply(lambda geom: line_merge(geom) if isinstance(geom, MultiLineString) else geom)
    centerlines_smoothed = centerlines_merged.simplify(tolerance=50, preserve_topology=True)
    gdf_centerlines_smoothed = gpd.GeoDataFrame(
        centerlines.drop(columns="geometry"),
        geometry=centerlines_smoothed,
        crs=centerlines.crs,
    )
    points = gdf_centerlines_smoothed.geometry.apply(lambda line: get_points_along_line(line, 1000))
    gdf_points = gpd.GeoDataFrame(
        geometry=[pt for sublist in points for pt in sublist], crs=gdf_centerlines_smoothed.crs
    )
    gdf_square_cutouts = gpd.GeoDataFrame(
        geometry=gdf_points.geometry.apply(lambda pt: pt.buffer(640).envelope), crs=gdf_points.crs
    )

    gdf_square_cutouts.to_parquet(
        f"../data/streets/depth_{depth}/streets_{code}_depth{depth}_boxes.parquet"
    )

11:13:15 WARNING  File for country ID does not exist, skipping.
11:13:15 WARNING  File for country MY does not exist, skipping.
11:13:15 WARNING  File for country CL does not exist, skipping.
11:13:15 WARNING  File for country BO does not exist, skipping.
11:13:15 WARNING  File for country PE does not exist, skipping.
11:13:15 WARNING  File for country AR does not exist, skipping.
11:13:15 WARNING  File for country -99 does not exist, skipping.
11:13:15 INFO     Boxes for country CY already exist, skipping.
11:13:15 INFO     Loaded 1 center-line segments for country CY.
11:13:15 WARNING  File for country IN does not exist, skipping.
11:13:15 WARNING  File for country CN does not exist, skipping.
11:13:15 WARNING  File for country IL does not exist, skipping.
11:13:15 WARNING  File for country PS does not exist, skipping.
11:13:15 WARNING  File for country LB does not exist, skipping.
11:13:15 WARNING  File for country ET does not exist, skipping.
11:13:15 WARNING  File for country SS d